In [7]:
import socket
import time
import xml.etree.ElementTree as ET
import re
from collections import defaultdict

class GilsonSession:
    def __init__(self, ip='192.168.1.101', initial_port=50185, timeout=3.0):
        self.ip = ip
        self.initial_port = initial_port
        self.timeout = timeout
        self.sock = None
        self.msg_id = 10000  # Start MessageId counter

    def connect(self):
        # Step 1: Connect to handshake port and send connect XML
        print(f"Connecting to handshake port {self.initial_port}...")
        handshake_sock = socket.create_connection((self.ip, self.initial_port), timeout=self.timeout)

        connect_xml = """
        <Gilson>
          <GilsonConnect>Gilson Ethernet Utility</GilsonConnect>
        </Gilson>
        """.strip()
        handshake_sock.sendall(connect_xml.encode('utf-8'))

        response = self._receive_response(handshake_sock)
        handshake_sock.close()

        # Step 2: Parse response to find command port (e.g., 50191)
        try:
            root = ET.fromstring(response)
            new_port = int(root.find('.//ParameterValue').text)
            print(f"Handshake complete. Switching to port {new_port}.")
        except Exception as e:
            raise RuntimeError(f"Failed to parse handshake response: {e}\nResponse: {response}")

        # Step 3: Open new connection to command port
        self.sock = socket.create_connection((self.ip, new_port), timeout=self.timeout)
        print(f"Connected to command port {new_port}.")

    def disconnect(self):
        if self.sock:
            self.sock.close()
            self.sock = None
            print("Disconnected.")

    def _make_message(self, device_name, command_name, device_id=None, sequence=0):
        # Create XML message with optional DeviceId
        device_info = f"<DeviceName>{device_name}</DeviceName>"
        if device_id is not None:
            device_info += f"<DeviceId>{device_id}</DeviceId>"

        command_xml = f"""
        <Message MessageId="{self.msg_id}" BlockCount="1" BlockId="0">
            <Gilson>
                <CommandData>
                    <Commands>
                        <Command>
                            <CommandInfo>
                                <InstrumentInfo>
                                    {device_info}
                                </InstrumentInfo>
                                <SequenceNumber>{sequence}</SequenceNumber>
                                <CommandXML>
                                    <CommandName>{command_name}</CommandName>
                                </CommandXML>
                            </CommandInfo>
                        </Command>
                    </Commands>
                </CommandData>
            </Gilson>
        </Message>
        """.strip()

        self.msg_id += 1
        return command_xml

    def send_xml_command(self, device_name, command_name, device_id=None):
        # Construct and frame the XML command
        xml_msg = self._make_message(device_name, command_name, device_id)
        encoded = xml_msg.encode('utf-8')
        length_prefix = len(encoded).to_bytes(4, byteorder='big')
        framed_msg = length_prefix + encoded

        # Send framed message
        self.sock.sendall(framed_msg)
        time.sleep(0.3)  # allow response time
        response = self._receive_response(self.sock)
        return self._parse_response(response)

    def _receive_response(self, sock, buffer_size=8192):
        # Read response until all data is received (we look for </Message>)
        chunks = []
        sock.settimeout(self.timeout)
        try:
            while True:
                chunk = sock.recv(buffer_size)
                if not chunk:
                    break
                chunks.append(chunk)
                if b'</Message>' in chunk:
                    break
        except socket.timeout:
            pass

        return b''.join(chunks).decode('utf-8', errors='replace')

    def _split_messages(self, xml_blob):
        """
        Reconstructs complete <Message> blocks from potentially fragmented responses.
        Only returns fully assembled messages.
        """
        raw_blocks = re.findall(r'(<Message.*?</Message>)', xml_blob, flags=re.DOTALL)
        grouped = defaultdict(dict)
        block_counts = {}

        for block in raw_blocks:
            try:
                msg_id = re.search(r'MessageId="(\d+)"', block).group(1)
                block_id = int(re.search(r'BlockId="(\d+)"', block).group(1))
                block_count = int(re.search(r'BlockCount="(\d+)"', block).group(1))

                grouped[msg_id][block_id] = block
                block_counts[msg_id] = block_count
            except Exception:
                continue

        # Only return complete reassembled messages
        reassembled = []
        for msg_id, blocks in grouped.items():
            expected = block_counts[msg_id]
            if len(blocks) == expected:
                ordered = [blocks[i] for i in sorted(blocks)]
                combined = ''.join(ordered)
                reassembled.append(combined)

        return reassembled

    def _parse_response(self, xml_response):
        messages = self._split_messages(xml_response)
        results = []

        for msg in messages:
            try:
                root = ET.fromstring(msg)
                cmd_name_el = root.find('.//CommandName')
                cmd_name = cmd_name_el.text if cmd_name_el is not None else "(Unknown)"
                params = root.findall('.//ReturnParameter')
                result = {
                    'Command': cmd_name,
                    'Result': {p.find('ParameterName').text: p.find('ParameterValue').text for p in params}
                }
                results.append(result)
            except Exception as e:
                results.append({'Error': f"Failed to parse a message block: {e}", 'Raw': msg})

        return results if len(results) > 1 else results[0]

    def __enter__(self):
        self.connect()
        return self

    def __exit__(self, *args):
        self.disconnect()


In [8]:
g = GilsonSession("192.168.1.101")
g.connect()

Connecting to handshake port 50185...
Handshake complete. Switching to port 2.
Connected to command port 2.


In [9]:
g.send_xml_command("System", "Admin")

{'Error': 'Failed to parse a message block: not well-formed (invalid token): line 1, column 4052',
 'Raw': '<Message MessageId="155" BlockCount="2" BlockId="0"><Gilson><Status><SystemInformation><IPAddress>192.168.1.101</IPAddress><SubnetMask>255.255.255.0</SubnetMask><DefaultGateway>192.168.1.1</DefaultGateway><MAC>00:40:9d:a2:56:92</MAC><LeaseInformation><LeaseActive>False</LeaseActive></LeaseInformation></SystemInformation><Devices><Device><DeviceName>GX-27x</DeviceName><DeviceId>35</DeviceId><SerialNumber>261J7N253</SerialNumber><Connected>True</Connected><Commands><Command><CommandName>Get Input Contacts</CommandName><Parameters><Parameter><ParameterName>ResponseCode</ParameterName><ParameterValue>2</ParameterValue></Parameter><Parameter><ParameterName>ResponseValue</ParameterName><ParameterValue><ReturnParameters><ReturnParameter><ParameterName>Contact A</ParameterName><ParameterValue>Off</ParameterValue></ReturnParameter><ReturnParameter><ParameterName>Contact B</ParameterName><

In [10]:
g.send_xml_command("System", "Get Firmware Version")

IndexError: list index out of range

In [ ]:
g.send_xml_command("GX-27x", "Identify", device_id=35)

In [6]:
g.send_xml_command("GX-27x", "Get Z Position", device_id=35)

[{'Command': '(Unknown)', 'Result': {}},
 {'Command': 'Get Z Position', 'Result': {}}]